# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [ ]:
# Task — Multi-series line with highlight
# Following the Lecture 3 notes: grey everything, highlight one, label directly, insight title

asia = df[df['Region'] == 'Asia']

# Choose a country to highlight
highlight = 'India'

# Numbers for the insight title
first_yr, last_yr = asia['Year'].min(), asia['Year'].max()
growth = (asia[asia['Year'] == last_yr].set_index('Country')['CO2_Mt'] /
          asia[asia['Year'] == first_yr].set_index('Country')['CO2_Mt'])
hl_growth = growth[highlight]
rank = int(growth.rank(ascending=False)[highlight])
story = 'the fastest growth in Asia' if rank == 1 else f'#{rank} of {len(growth)} Asian countries for growth'
title = f"{highlight}'s CO2 emissions grew {hl_growth:.1f}x from {first_yr} to {last_yr} — {story}"

# Build colour map: highlight in colour, everything else grey (#EEEEEE)
color_map = {c: 'crimson' if c == highlight else '#EEEEEE' for c in asia['Country'].unique()}

fig = px.line(asia, x='Year', y='CO2_Mt', color='Country',
              color_discrete_map=color_map,
              labels={'CO2_Mt': 'CO2 Emissions (Mt)', 'Year': ''},
              title=title,
              # Gestalt Closure: white background removes the need for a chart border
              template='plotly_white',
              # Gestalt Connection: no markers needed
              markers=False,
              height=450)

fig.update_traces(
    line=dict(width=1.5),   # default: thin for context countries
    showlegend=False        # direct label replaces legend (Gestalt proximity)
)

# Override highlighted country: thicker line to draw attention
fig.update_traces(
    line=dict(width=3),
    selector=dict(name=highlight)  # each line is named after a country
)

# Draw the highlighted line on top of the grey ones
fig.data = tuple(sorted(fig.data, key=lambda t: t.name == highlight))

# Direct label at end of highlighted line
highlight_label = asia.loc[(asia['Country'] == highlight) & (asia['Year'] == last_yr)]
fig.add_annotation(
    x=highlight_label['Year'].values[0], y=highlight_label['CO2_Mt'].values[0],
    text=highlight, showarrow=False,
    xanchor='left', xshift=5,
    font=dict(color='crimson', size=13, weight='bold', family='Arial')
)

# Light labels for the context countries (optional)
for c in asia['Country'].unique():
    if c == highlight:
        continue
    row = asia.loc[(asia['Country'] == c) & (asia['Year'] == last_yr)]
    fig.add_annotation(
        x=row['Year'].values[0], y=row['CO2_Mt'].values[0],
        text=c, showarrow=False, xanchor='left', xshift=5,
        font=dict(color='#BBBBBB', size=11, family='Arial')
    )

fig.update_layout(
    font=dict(size=13),
    # Gestalt Continuity: very light gridlines so the data stands out
    yaxis=dict(gridcolor='#EEEEEE', rangemode='tozero', tickformat=','),
    xaxis=dict(showgrid=False, tickangle=0),   # no rotated text
    margin=dict(r=110)                          # room for the direct labels
)

fig.show()
